# Day 5 模块 3：模型帮经营决策

模型不是为了「秀分」，而是：

1. **哪些因素更牵动预测中的营收**（特征重要性）
2. **假如经营条件变了，模型会猜多少**（what-if）

注意：重要性 ≠ 严格因果；what-if 是「按历史规律外推」，不是保证。


In [ ]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path('day5_cafe_sales.csv'),
    Path('day5') / 'day5_cafe_sales.csv',
    Path('教学课程') / 'day5' / 'day5_cafe_sales.csv',
]
for path in candidate_paths:
    if path.exists():
        csv_path = path
        break
else:
    raise FileNotFoundError('未找到 day5_cafe_sales.csv')

df = pd.read_csv(csv_path)
print(csv_path.resolve())
print('shape:', df.shape)
df.head()


In [ ]:
# 与 Day 3/4 同一套经营特征，方便接模型
weather_score_map = {'晴': 1.0, '多云': 0.8, '阴': 0.6, '小雨': 0.4, '大雨': 0.3}
df = df.copy()
df['weather_score'] = df['weather_label'].map(weather_score_map)

# 经营场景标签（给人看的，不是给模型背的）
df['weekend_label'] = df['is_weekend'].map({0: '工作日', 1: '周末'})
df['promo_label'] = df['promotion'].map({0: '无促销', 1: '有促销'})

print(df[['day', 'weather_label', 'weekend_label', 'promo_label', 'sales']].head())
print('平均营收:', round(df['sales'].mean(), 1))


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import pandas as pd

feature_cols = [
    'price', 'promotion', 'is_weekend', 'temp', 'quality',
    'competitors', 'holiday', 'location', 'weather_score',
]
X = df[feature_cols]
y = df['sales']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf = RandomForestRegressor(
    n_estimators=100, max_depth=8, random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
pred = rf.predict(X_test)
print('测试 R² :', round(r2_score(y_test, pred), 3))
print('测试 MAE:', round(mean_absolute_error(y_test, pred), 1))


## 1. 特征重要性 → 业务话


In [ ]:
imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(imp.round(3))
print()
print('Top3:', list(imp.head(3).index))


### 怎么跟店长说（模板）

```text
在当前这批历史数据上，模型做预测时更依赖：A、B、C。
这提示我们日常复盘时，要优先盯这几项的变化。
但「依赖高」不等于「改它就一定涨」——还要结合现场经验。
```


## 2. What-if：基准日 vs 改一个条件

先固定一组「基准经营条件」，再只改一个旋钮，看预测营收变化。


In [ ]:
# 基准：晴天分、工作日、无促销、中等价与品质
base = {
    'price': 25.0,
    'promotion': 0,
    'is_weekend': 0,
    'temp': 22.0,
    'quality': 7.5,
    'competitors': 2,
    'holiday': 0,
    'location': 7.0,
    'weather_score': 1.0,
}

def predict_one(overrides=None):
    row = base.copy()
    if overrides:
        row.update(overrides)
    x = pd.DataFrame([row], columns=feature_cols)
    return round(float(rf.predict(x)[0]), 1)

print('基准预测营收:', predict_one())
print('改为有促销  :', predict_one({'promotion': 1}))
print('改为周末    :', predict_one({'is_weekend': 1}))
print('竞品+2      :', predict_one({'competitors': base['competitors'] + 2}))
print('天气变大雨  :', predict_one({'weather_score': 0.3}))
print('单价+5      :', predict_one({'price': base['price'] + 5}))


## 3. 读 what-if 的规矩

| 可以 | 小心 |
| --- | --- |
| 比较「相对谁更高/更低」 | 把预测当成明天的准账 |
| 用来排「先试什么」 | 一次改十个条件说不清 |
| 结合场景分析一起讲 | 忽略 MAE：预测本身有误差带 |

## 4. 小练习

1. Top3 重要性用业务话写三句。
2. 你做的 what-if 里，对营收伤害最大的单一变化是哪个？
3. 若只能给店长一个「本周实验」，你建议试什么？如何用数据验证？


In [ ]:
# 在这里写
